# Ejercicio 1: Evaluación de solicitud de crédito bancario

Diseño de un agente inteligente que evalúa solicitudes de crédito a partir del ingreso mensual, el monto solicitado y el historial de morosidad del cliente.

### Ficha PEAS

| Componente | Descripción |
|---|---|
| **P — Medida de desempeño** | Minimizar el riesgo de impago sin rechazar solicitudes solventes; mantener decisiones coherentes y justificables. |
| **E — Entorno** | Sistema bancario de evaluación de solicitudes, datos financieros del solicitante y políticas de riesgo de la entidad. |
| **A — Acciones** | Aprobar, aprobar con condiciones o rechazar la solicitud de crédito. |
| **S — Percepciones** | Ingreso mensual, monto solicitado e indicador de historial moroso. |

**Objetivo:** determinar una respuesta adecuada para cada solicitud, equilibrando el acceso al crédito con la reducción del riesgo financiero para el banco.

### Justificación de las reglas

Se aplicarán las siguientes reglas de decisión:

- Si la relación monto/ingreso es menor o igual a 0.5 y no existe historial moroso, se aprueba la solicitud.
- Si la relación es menor o igual a 0.5 pero existe historial moroso, se aprueba con condiciones.
- Si la relación es mayor a 0.5 y menor o igual a 1, se aprueba con condiciones únicamente cuando no existe historial moroso; en caso contrario, se rechaza.
- Si la relación es mayor a 1, se rechaza la solicitud independientemente del historial.

Una relación de hasta 0.5 representa una deuda más manejable frente al ingreso mensual y, sin morosidad previa, implica menor riesgo.
El historial moroso eleva la probabilidad de incumplimiento, por lo que exige condiciones adicionales o provoca el rechazo en relaciones altas.
Cuando el monto supera el ingreso mensual, el compromiso financiero se considera demasiado elevado para aprobar la solicitud.

In [ ]:
def agente_credito(ingreso_mensual, monto_solicitado, tiene_historial_moroso):
    """Evalúa una solicitud y retorna una tupla (acción, motivo)."""
    if ingreso_mensual <= 0:
        raise ValueError("El ingreso mensual debe ser mayor que cero.")
    if monto_solicitado < 0:
        raise ValueError("El monto solicitado no puede ser negativo.")
    if not isinstance(tiene_historial_moroso, bool):
        raise TypeError("El historial moroso debe ser True o False.")

    relacion = monto_solicitado / ingreso_mensual

    if relacion <= 0.5:
        if tiene_historial_moroso:
            return (
                "aprobar con condiciones",
                f"relación monto/ingreso de {relacion:.2f}, pero presenta historial moroso",
            )
        return (
            "aprobar",
            f"relación monto/ingreso de {relacion:.2f} y sin historial moroso",
        )

    if relacion <= 1:
        if tiene_historial_moroso:
            return (
                "rechazar",
                f"relación monto/ingreso de {relacion:.2f} con historial moroso",
            )
        return (
            "aprobar con condiciones",
            f"relación monto/ingreso de {relacion:.2f}, aunque no presenta historial moroso",
        )

    return (
        "rechazar",
        f"relación monto/ingreso de {relacion:.2f}, superior al límite de 1.00",
    )

### Simulación y pruebas

Se evalúan ocho solicitudes que cubren las tres acciones posibles, ambos valores del historial y los límites exactos de relación `0.50` y `1.00`.

In [ ]:
casos_prueba = [
    # ingreso, monto, historial moroso, descripción
    (4000, 1200, False, "relación baja sin morosidad"),
    (4000, 1200, True, "relación baja con morosidad"),
    (4000, 2000, False, "límite 0.50 sin morosidad"),
    (4000, 2000, True, "límite 0.50 con morosidad"),
    (4000, 3000, False, "relación intermedia sin morosidad"),
    (4000, 3000, True, "relación intermedia con morosidad"),
    (4000, 4000, False, "límite 1.00 sin morosidad"),
    (4000, 4800, False, "relación mayor a 1.00"),
]

for numero, (ingreso, monto, historial, descripcion) in enumerate(casos_prueba, start=1):
    accion, motivo = agente_credito(ingreso, monto, historial)
    relacion = monto / ingreso
    print(f"Caso {numero}: {descripcion}")
    print(f"  Entrada: ingreso=S/ {ingreso}, monto=S/ {monto}, moroso={historial}")
    print(f"  Relación: {relacion:.2f} | Decisión: {accion}")
    print(f"  Motivo: {motivo}\n")

### Visualización

Se generan 200 solicitudes aleatorias y se representa cada una según su relación monto/ingreso, su ingreso mensual y la decisión tomada por el agente. Se utiliza una semilla fija para que la simulación sea reproducible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
cantidad_solicitudes = 200

ingresos = rng.integers(1200, 10001, size=cantidad_solicitudes)
montos = rng.integers(500, 12001, size=cantidad_solicitudes)
historiales_morosos = rng.choice(
    [False, True], size=cantidad_solicitudes, p=[0.75, 0.25]
)
relaciones = montos / ingresos

acciones = np.array([
    agente_credito(int(ingreso), int(monto), bool(historial))[0]
    for ingreso, monto, historial in zip(ingresos, montos, historiales_morosos)
])

colores = {
    "aprobar": "#2ca02c",
    "aprobar con condiciones": "#ff9800",
    "rechazar": "#d62728",
}

plt.figure(figsize=(10, 6))
for accion, color in colores.items():
    seleccion = acciones == accion
    plt.scatter(
        relaciones[seleccion],
        ingresos[seleccion],
        c=color,
        label=f"{accion} ({seleccion.sum()})",
        alpha=0.7,
        edgecolors="white",
        linewidths=0.4,
    )

plt.axvline(0.5, color="gray", linestyle="--", linewidth=1, label="límite 0.50")
plt.axvline(1.0, color="black", linestyle="--", linewidth=1, label="límite 1.00")
plt.title("Decisiones del agente para 200 solicitudes de crédito")
plt.xlabel("Relación monto solicitado / ingreso mensual")
plt.ylabel("Ingreso mensual (S/)")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()